In [1]:
import argparse
from pathlib import Path
from typing import Dict
import os
import json

import torch
import torch.distributed
from finetuning import CheckpointManager

from unakin.llm.adapters.llm_adapter import LlmAdapter
from unakin.llm.dialogue.base import DialogueModel
from unakin.llm.models.utils import ModelFactory
from unakin.llm.tokenizers import LlamaTokenizer
from unakin.llm.tokenizers.base import TokenizerBase

/home/fsuser/miniconda3/envs/ncs-stable/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from typing import Optional

REFERENCE_MAP = {
  "13B": {
    "model": {
        "type": "llama",
        "args": {
          "base": "code-llama-13b-instruct"
        }
      },

        "type": "llama",
        "path": "/mnt/raid0/llamas/codellama/CodeLlama-13b-Instruct/tokenizer.model"
      },
  "34B": {
    "model": {
        "type": "llama",
        "args": {
          "base": "code-llama-34b-instruct"
        }
      },
      "tokenizer": {
        "type": "llama",
        "path": "/mnt/raid0/llamas/codellama/CodeLlama-34b-Instruct/tokenizer.model"
      }
  }
}

def get_ckpt(path: str, skip_path: bool = True) -> Dict:
  if skip_path:
    return {}

  ckpt = torch.load(f"{path}/ckpt.pt", map_location="cpu")
  return ckpt

def _get_dev_model(reference_type: str = "13B") -> DialogueModel:
  model, tokenizer = ModelFactory.from_json(REFERENCE_MAP[reference_type])

  model.enable_xops(enable=True)
  model.to(torch.cuda.current_device())

  # model.to(torch.cuda.current_device())

  return DialogueModel(model=model, tokenizer=tokenizer)


def get_dialogue_model_from_ckpt(
    ckpt_root: str, ckpt_name: str, ckpt_step: str,
    reference_type: Optional[str] = None, batch_size: int = 1) -> DialogueModel:
  
  path = Path(ckpt_root) / ckpt_name / ckpt_step / "ckpt.pth"

  if reference_type is not None:
    print(f"Using reference type: {reference_type}")
    model = _get_dev_model(reference_type=reference_type)
    model.model.register_cache_kv(max_bsz=batch_size, max_seq_len=4096)
    return model

  ckpt_manager = CheckpointManager(path)
  
  # model
  model, tokenizer = ckpt_manager.init_model_from_ckpt(path=path, root_path=ckpt_root)
  model: LlmAdapter

  # move model to device
  model = model.to(torch.cuda.current_device())
  model = model.half()

  # merge adaptation
  model.merge_adaptation()
  model.register_cache_kv(max_bsz=batch_size, max_seq_len=4096)

  # create the dialogue model
  dialogue_model = DialogueModel(model=model, tokenizer=tokenizer)

  return dialogue_model

def read_json_lines(path: str):
  assert path.endswith(".jsonl")
  with open(path, "r") as f:
    lines = f.readlines()
    lines = [json.loads(line) for line in lines]
    lines = [line["task"] for line in lines]
    return lines

def read_system_prompt(path: str):
  if not path:
    return None

  with open(path, "r") as f:
    return f.read().strip()
  
def get_pad_dialogue(batch: list[list], target_size: int = 16, add_system_prompt: bool = True):
  pad = [{"role": "user", "content": "blank"}]
  if add_system_prompt:
    pad = [{"role": "system", "content": "blank"}] + pad
  
  for _ in range(target_size - len(batch)):
    batch.append(pad)
    
  return batch  

def make_dialogue_message(content: str, system_prompt: str | None):
  user_message = {"role": "user", "content": content}
  if system_prompt is None:
    return [user_message]
  
  system_message = {"role": "system", "content": system_prompt}
  return [system_message, user_message]


def init_torch(args):
  if args.world_size > 1:
    init_torch_distributed(args)
    rank = torch.distributed.get_rank()
    device = torch.device(f"cuda:{rank}")
  else:
    device = torch.device(f"cuda:{args.device_id}")
  torch.cuda.set_device(device)
  torch.set_default_dtype(torch.float16)

def init_torch_distributed(args):
  # init_torch(args)
  if args.world_size > 1:
    torch.distributed.init_process_group(
      backend="nccl",
      init_method="env://",
    )

def make_dialogue_message(content: str, system_prompt: str | None):
  user_message = {"role": "user", "content": content}
  if system_prompt is None:
    return [user_message]
  
  system_message = {"role": "system", "content": system_prompt}
  return [system_message, user_message]

In [3]:
import argparse

args = argparse.Namespace(
  world_size=1,
  device_id=0,
  ckpt_root="/mnt/raid0/checkpoints",
  ckpt_name="bigmix_ft_fim_rev",
  ckpt_step="004608",
  reference_type=None,
)

In [4]:
try:
  del dialogue_model
  torch.cuda.empty_cache()
except:
  pass

init_torch(args)
dialogue_model = get_dialogue_model_from_ckpt(
      ckpt_root=args.ckpt_root,
      ckpt_name=args.ckpt_name,
      ckpt_step=args.ckpt_step,
      reference_type=args.reference_type,
  )

/mnt/raid0/llamas/llama_base/code-llama-13b-instruct.pth code-llama-13b-instruct
/mnt/raid0/llamas/llama_base/code-llama-13b-instruct.pth
Allocated virtual model in 0.02351522445678711s
Loaded virtual model in 0.006514787673950195s
XxxXXXXXXXXXXX Tokenizer path: /mnt/raid0/llamas/codellama/CodeLlama-13b-Instruct/tokenizer_with_rev.model
Loading tokenizer from /mnt/raid0/llamas/codellama/CodeLlama-13b-Instruct/tokenizer_with_rev.model
Adding reverse embedding
INFO: Registering cache of size 1 x 4096


In [7]:
import pprint

instruction1 = """Create a C# Script for a simple jump action in a character using raycast that moves in a 2D environment in Unity."""
response1 = """Okay, I'll do my best! Here's a class for a Jump action in a character using ray tracing:
```cs
using UnityEngine;
using System.Collections;
using System.Collections.Generic;

public class Jump: MonoBehaviour
{
    public float jumpHeight = 5f;
    public float groundDistance = 0.1f; // how far the ray will check for the ground
    public LayerMask groundLayer; // which layers are the ground. Set this in unity

    private Rigidbody2D rb;

    void Start() // init
    {
        rb = GetComponent<Rigidbody2D>();
    }

    void Update()
    {
        if (IsGrounded() && Input.GetKeyDown(KeyCode.Space)) // if character is on the ground and jumps
        {
            rb.velocity = new Vector2(rb.velocity.x, jumpHeight); // make them jump!
        }
    }

    bool IsGrounded()
    {
        RaycastHit2D raycastHit = Physics2D.Raycast(transform.position, Vector2.down, groundDistance, groundLayer); // create raycast pointing to the ground
        return raycastHit.collider != null; // if the ray hit ground, user is on ground
    }
}
```"""

instruction2 = """Can you write me a script in unity using the rigidbody component for first-person movement in a threedimensional game?"""
response2 = """```csharp
using UnityEngine;

public class FirstPersonMovement : MonoBehaviour {
	// Rigidbody component used to move the character
	public Rigidbody rb;

	// Use this function to move the character in first-person mode
	void Update() {
		float horizontal = Input.GetAxis("Horizontal");
		float vertical = Input.GetAxis("Vertical");

		// Calculate new position based on input
		Vector3 newPosition = new Vector3(horizontal, 0, vertical);

		// Set the new position of the character’s Rigidbody
		rb.transform.position = newPosition;
	}
}
```
This script uses the Input.GetAxis() function to get the horizontal and vertical input from the user, and uses the newPosition variable to set the new position of the character's Rigidbody. The rb component is the instance of the Rigidbody component that we want to move, and the transform property is the location of the character. Finally, the Update() function is called every frame to update the character's position."""

instruction3 = """How would I make a mod for Vallheim that made it so that you had a skill level threshold you couldn't go under when you died.   Otherwise you would loose levels as normal when you died.   For example if you hit level 10 in wood cutting, it would be the new threshold.   If you hit level 20 that would then become the new threshold, until level 85."""
response3 = """Here's an example of how you could implement the skill level threshold feature in a Valheim mod:

First, you will need to find the relevant code in the game that handles the player's skill levels and death. In Valheim, player skills are stored in the player's character file, which is a binary file that contains information about the player's character, including their skills and stats.

Once you have found the relevant code, you can create a new script or modify an existing one to implement the skill level threshold feature. Here's an example script that demonstrates how you could achieve this:
```csharp
// Import the necessary Valheim classes and namespaces
using UnityEngine;
using System.IO;

// Define the SkillLevelThreshold mod class
public class SkillLevelThreshold : MonoBehaviour
{
    // The minimum skill level threshold
    private int minThreshold = 10;

    // The maximum skill level threshold
    private int maxThreshold = 85;

    // The path to the player's character file
    private string characterFilePath = "";

    // The main mod entry point
    void Start()
    {
        // Get the path to the player's character file
        characterFilePath = Path.Combine(Application.persistentDataPath, "characters", "player.fch");

        // Load the player's character file into a byte array
        byte[] characterData = File.ReadAllBytes(characterFilePath);

        // Get the player's skill levels from the character file
        int[] skillLevels = new int[10];
        for (int i = 0; i < 10; i++)
        {
            skillLevels[i] = BitConverter.ToInt32(characterData, 152 + i * 4);
        }

        // Find the highest skill level achieved by the player
        int maxSkillLevel = 0;
        foreach (int level in skillLevels)
        {
            if (level > maxSkillLevel)
            {
                maxSkillLevel = level;
            }
        }

        // Set the minimum threshold based on the highest skill level achieved
        if (maxSkillLevel > minThreshold)
        {
            minThreshold = maxSkillLevel;
        }

        // Clamp the minimum threshold to the maximum threshold
        minThreshold = Mathf.Min(minThreshold, maxThreshold);
    }

    // Handle the player's death event
    void OnPlayerDead()
    {
        // Get the player's skill levels from the character file
        byte[] characterData = File.ReadAllBytes(characterFilePath);
        int[] skillLevels = new int[10];
        for (int i = 0; i < 10; i++)
        {
            skillLevels[i] = BitConverter.ToInt32(characterData, 152 + i * 4);
        }

        // Check if any of the player's skill levels are below the threshold
        for (int i = 0; i < 10; i++)
        {
            if (skillLevels[i] < minThreshold)
            {
                // Reset the skill level to the threshold
                skillLevels[i] = minThreshold;
            }
        }

        // Write the updated skill levels back to the character file
        for (int i = 0; i < 10; i++)
        {
            Buffer.BlockCopy(BitConverter.GetBytes(skillLevels[i]), 0, characterData, 152 + i * 4, 4);
        }
        File.WriteAllBytes(characterFilePath, characterData);
    }
}
```
This script works by finding the player's skill levels in their character file, determining the highest skill level achieved, and setting the minimum threshold based on that value. When the player dies, the script checks if any of their skill levels are below the threshold, and resets them to the threshold if necessary."""

# system_prompt = f"You are a helpfull engineering assistant. You design problem descriptions and instructions for generating code snniptes. You use only natural language and do not respond with code."
# instruction = f"Using the provided code snippet as a reference, create a comprehensive problem description that includes necessary details. Ensure that the problem is clearly defined and highlights any constraints. The problem description should be detailed enough for an engineer to understand and implement a solution:\n```\n{code}```"
# instruction = f"\n{code}\n  Based on the code reference provided, develop a comprehensive problem statement."
# instruction = f"Generate an instruction for the following code:\n{code}\n"

# instruction = \
# f"""
# EXAMPLE 1:
# INSTRUCTION: {instruction1}
# OUTPUT: {response1}
# CONSTRAINTS: None

# EXAMPLE 2:
# INSTRUCTION: {instruction2}
# OUTPUT: {response2}
# CONSTRAINTS: None

# EXAMPLE 3:
# INSTRUCTION: {instruction3}
# OUTPUT: {response3}
# CONSTRAINTS: None

# EXAMPLE 4:
# """


instruction = \
f"""
EXAMPLE 1:
INSTRUCTION: {instruction1}
OUTPUT: {response1}
CONSTRAINTS: None

EXAMPLE 2:
INSTRUCTION: {instruction2}
OUTPUT: {response2}
CONSTRAINTS: None

From now on, the examples will be functionally diverse and used for developing games with dragons, goblins, cowboys and broads. Do not repeat fuctionalities from previous examples.

EXAMPLE 3:
INSTRUCTION: """

# instruction = "Provide a sequence of distinct scripting instructions for developing a shooter game using Unity-C# without referencing interactions with Unity's editor or any external software. Assume I'm already in my IDE, prepared to write code, and omit instructions related to file creation or setup. Each point should detail the task for a separate script, without combining multiple tasks into one script. List the scripting steps required to build the game's core mechanics."

# instruction = "Who is Teodor Poncu?"

# message = make_dialogue_message(
#     content=instruction, system_prompt=None
#   )

# prompt_prediction = dialogue_model.chat_completion(
#   [message], temperature=0.6, top_p=0.9, max_gen_len=2048, repeat_penalty=1.0)
# print(prompt_prediction[0]['generation']["content"])

prompt_prediction = dialogue_model.text_completion(
    [instruction], temperature=0.6, top_p=0.9, max_gen_len=1024, repeat_penalty=1.0)
# pprint.pprint(prompt_prediction)
print(prompt_prediction[0]['generation'])

Generating 1 sequences of max length 1766 on cuda:0


100%|██████████| 1024/1024 [00:23<00:00, 44.01it/s]


Write a script that will make a character jump when the space bar is pressed, and apply a force to the character's Rigidbody to make them move in the direction they are facing.

OUTPUT:
```csharp
using UnityEngine;

public class JumpAndMove : MonoBehaviour {
	// Rigidbody component used to move the character
	public Rigidbody rb;

	// Use this function to move the character in first-person mode
	void Update() {
		float horizontal = Input.GetAxis("Horizontal");
		float vertical = Input.GetAxis("Vertical");

		// Calculate new position based on input
		Vector3 newPosition = new Vector3(horizontal, 0, vertical);

		// Set the new position of the character’s Rigidbody
		rb.transform.position = newPosition;

		// Check to see if the user has pressed the space bar
		if (Input.GetKeyDown(KeyCode.Space)) {
			// Add a force to the character's Rigidbody
			rb.AddForce(Vector3.up * 500);
		}
	}
}
```
This script uses the Input.GetAxis() function to get the horizontal and vertical input from the

In [8]:
import pandas as pd
import pprint
from typing import NamedTuple

class Sample(NamedTuple):
  description: str
  code: str

class DiscriminativeSample(NamedTuple):
  generated_instruction: str
  code: str

df = pd.read_json("all_data.json", orient="records")
df = df[df["Codes_Count"] < 2]

samples = []
for idx, row in df.iterrows():
  code = row["Codes"][0]
  code = code.replace("\r\n", "\n")
  code = f"```csharp\n{code}\n```"
  samples.append(Sample(description=row["Prompt"], code=code))

discriminative_samples = []
descriptions = []
code_samples = []

for s in samples[:10]:
  code = s.code
  system_prompt = None # f"You are a helpfull engineering assistant. You design problem descriptions and instructions for generating code snniptes. You use only natural language and never provide code. Specifficaly respect all high-level functionalities from the presented code."
  instruction = f"Describe the following code: \n{code}"

  code_samples.append(code)
  
  description = dialogue_model.chat_completion(
    [make_dialogue_message(content=instruction, system_prompt=system_prompt)],
    temperature=0.6, top_p=0.9, max_gen_len=2048, repeat_penalty=1.)
  
  discriminative_samples.append(DiscriminativeSample(
    generated_instruction=description[0]["generation"]["content"],
    code=code
  ))

  descriptions.append(description[0]["generation"]["content"])
  
  pprint.pprint(description)
  print(s.description)
  print("*" * 80)


Generating 1 sequences of max length 3156 on cuda:0


  0%|          | 0/2048 [00:00<?, ?it/s]

 53%|█████▎    | 1078/2048 [00:24<00:22, 43.54it/s]


[{'generation': {'content': '```csharp\n'
                            'using UnityEngine;\n'
                            'using UnityEngine.UI;\n'
                            'using TMPro;\n'
                            '\n'
                            'public class PauseUI : MonoBehaviour\n'
                            '{\n'
                            '    [SerializeField]\n'
                            '    Button _pauseButton;\n'
                            '    [SerializeField]\n'
                            '    Button _resumeButton;\n'
                            '    [SerializeField]\n'
                            '    TextMeshProUGUI _pauseText;\n'
                            '    [SerializeField]\n'
                            '    const string PAUSE_TEXT = "PAUSED";\n'
                            '    [SerializeField]\n'
                            '    const string PLAYING_TEXT = "PLAYING";\n'
                            '\n'
                            '    public bool gen

 81%|████████  | 1657/2048 [00:38<00:09, 42.89it/s]


...

[{'generation': {'content': '```csharp\n'
                            'using System.Collections;\n'
                            'using System.Collections.Generic;\n'
                            'using UnityEngine;\n'
                            'using UnityEngine.UI;\n'
                            'using TMPro;\n'
                            '\n'
                            'public class ObjectDestroyButtons : MonoBehaviour '
                            '{\n'
                            '\n'
                            '    public List<GameObject> objectList = new '
                            'List<GameObject>();\n'
                            '    private void Start()\n'
                            '    {\n'
                            '        if(objectList.Count == 0)\n'
                            '        {\n'
                            '            Debug.LogError("Please add objects to '
                            'the list!");\n'
                            '            retur

 34%|███▎      | 688/2048 [00:15<00:30, 44.15it/s]


[{'generation': {'content': '```csharp\n'
                            'using UnityEngine;\n'
                            'using UnityEngine.UI;\n'
                            'using TMPro;\n'
                            '\n'
                            'public class SimplePanel : MonoBehaviour\n'
                            '{\n'
                            '    public bool generateUI = true;\n'
                            '\n'
                            '    public Button closeButton;\n'
                            '\n'
                            '    private void Start()\n'
                            '    {\n'
                            '        if(generateUI)\n'
                            '        {\n'
                            '            GenerateUI();\n'
                            '        }\n'
                            '    }\n'
                            '\n'
                            '    // Method to generate UI\n'
                            '    private void Ge

 53%|█████▎    | 1093/2048 [00:25<00:21, 43.58it/s]

[{'generation': {'content': '```csharp\n'
                            'using System.Collections;\n'
                            'using System.Collections.Generic;\n'
                            'using UnityEngine;\n'
                            'using UnityEngine.UI;\n'
                            'using TMPro;\n'
                            '\n'
                            'public class PanelCloseTween : MonoBehaviour {\n'
                            '\n'
                            '    public bool generateUI = true;\n'
                            '    // Close button, the rect of the panel and '
                            'the starting size\n'
                            '    public Button closeButton;\n'
                            '    public RectTransform panelRect;\n'
                            '    private Vector2 startRect;\n'
                            '\n'
                            '    public float animSpeed = 0.01f;\n'
                            '\n'
                

In [7]:
# try:
#   del discriminator_dialogue_model
# except Exception as e:
#   pass

# args_discriminator = argparse.Namespace(
#   world_size=1,
#   device_id=2,
#   ckpt_root="/home/ted/workdir/ncs/ml-research/recipes",
#   ckpt_name="checkpoints/l4_instr_fw_20M",
#   ckpt_step="001280",
#   reference_type="34B",
# )

# init_torch(args_discriminator)

# discriminator_dialogue_model = get_dialogue_model_from_ckpt(
#       ckpt_root=args_discriminator.ckpt_root,
#       ckpt_name=args_discriminator.ckpt_name,
#       ckpt_step=args_discriminator.ckpt_step,
#       reference_type=args_discriminator.reference_type,
# )

# discriminator_dialogue_model.model.register_cache_kv(max_bsz=1, max_seq_len=8192)

In [9]:
humpback_template = \
"""Below is an instruction from an user and a response. Evaluate whether or not the response is a good example of how AI Assistant should respond to the user’s instruction. Please assign a score using the following 5-point scale:
1: The code is incomplete, vague, off-topic, or not what the user asked for. It may be missing content, have organizational issues, or repeat the user's question. It might also contain irrelevant or controversial information.
2: The code addresses most of the user's asks but does not directly answer the question. It may provide a high-level methodology instead of an exact solution.
3: The code is helpful, but it may be written with a more human perspective. It covers the basics and might contain explanations or context that could be useful.
4: The code provides a clear, comprehensive response to the user's question. It is well-organized, self-contained, and written in a helpful manner. There may be minor room for improvement, like being more concise.
5: The code is exemplary. It is focused, well-written, logical, easy to follow, and engages the user with expert knowledge in the area. It may also include highly relevant and insightful comments.
Do not repeat the input, instruction or instruction response. Please first provide a brief reasoning you used to derive the rating score, and then write "Score: <rating>" in the last line.

### INSTUCTION 
{generated_instruction}

### RESPONSE 
{output}

### REASONING
"""

reasoining_scores = []

for s in discriminative_samples:
  instruction = s.generated_instruction
  output = f"Sure, here is a piece of code that implements the desired functionality:\n{s.code}"
  sample_instruction = humpback_template.format(
    generated_instruction=instruction, output=output
  )
  message = make_dialogue_message(
    content=sample_instruction, system_prompt=None
  )
  prompt_prediction = dialogue_model.chat_completion(
    [message], temperature=0.6, top_p=0.9, max_gen_len=2048, repeat_penalty=1.0)
  #pprint.pprint(sample_instruction)
  pprint.pprint(prompt_prediction)
  print("*" * 80)

  reasoining_scores.append(prompt_prediction[0]['generation']["content"])

Generating 1 sequences of max length 4583 on cuda:0


  4%|▍         | 81/2048 [00:02<00:55, 35.76it/s]


[{'generation': {'content': 'This code is a great example of how to create a '
                            'pause menu in Unity. It creates a vertical layout '
                            'group and adds buttons and a text element to it. '
                            'It also adds listeners to the buttons and sets '
                            'the default text. This code is well-organized, '
                            'easy to follow, and includes helpful comments. It '
                            'covers the basics and might contain explanations '
                            'or context that could be useful. ',
                 'role': 'assistant'}}]
********************************************************************************
Generating 1 sequences of max length 5741 on cuda:0


  9%|▉         | 182/2048 [00:04<00:49, 37.71it/s]


[{'generation': {'content': 'This code is a good start, but there are a few '
                            'things that could be improved. \n'
                            '\n'
                            'First, the code could be more organized. For '
                            'example, the GenerateUI method could be broken up '
                            'into smaller methods, each with a specific '
                            'purpose. This would make the code easier to read '
                            'and understand.\n'
                            '\n'
                            'Second, the code could be more efficient. For '
                            'example, the CreateButton method could be '
                            'simplified by using the '
                            'Button.onClick.AddListener method instead of '
                            'adding a new event listener every time the button '
                            'is clicked.\n'
                           

...

[{'generation': {'content': 'The code provided is a good start for creating a '
                            'UI that can destroy game objects. It creates '
                            'buttons for each game object in a list, and adds '
                            'a listener to each button that removes the object '
                            'from the list, destroys the object, and destroys '
                            'the button. \n'
                            '\n'
                            'There are a few minor improvements that could be '
                            'made to the code. First, the code could be more '
                            'efficient by using a foreach loop instead of a '
                            'for loop. This would avoid the need to create a '
                            'separate index variable and would simplify the '
                            'code.\n'
                            '\n'
                            'Second, the code could be more 

  4%|▍         | 89/2048 [00:02<00:49, 39.24it/s]


[{'generation': {'content': 'The code is well-organized, self-contained, and '
                            'written in a helpful manner. It covers the basics '
                            'and might contain explanations or context that '
                            'could be useful. The code provides a clear, '
                            "comprehensive response to the user's question. It "
                            'is well-written, logical, easy to follow, and '
                            'engages the user with expert knowledge in the '
                            'area. It may also include highly relevant and '
                            'insightful comments. ',
                 'role': 'assistant'}}]
********************************************************************************
Generating 1 sequences of max length 4616 on cuda:0


  8%|▊         | 171/2048 [00:04<00:47, 39.64it/s]

[{'generation': {'content': 'The code is well-written, but it could be more '
                            'concise. For example, the `GenerateUI` method '
                            'could be refactored to use the `CreateButton` '
                            'method to create the close button, instead of '
                            'creating it twice. The `ShrinkCloseCoroutine` '
                            'method could also be simplified by using the '
                            '`Lerp` method to calculate the current size of '
                            'the panel instead of using a separate `t` '
                            'variable.\n'
                            '\n'
                            'Additionally, the code could be improved by using '
                            'more descriptive variable names, such as '
                            '`panelSize` instead of `startRect`. This would '
                            'make the code more readable and easier to '
        

'The answer provides a complete code solution to '
"the user's instruction. The code includes the "
'requested properties and methods, and the code is '
'well-organized and easy to follow. The code is '
'also well-written and logical, and it engages the '
'user with expert knowledge in the area.\n'
'\n'
'The code provides a clear, comprehensive response '
"to the user's question. It is well-organized, "
'self-contained, and written in a helpful manner. '
'The code is focused, well-written, and easy to '
'follow.\n'
'\n'
'Therefore, the answer is a good example of how AI '
"Assistant should respond to the user's "
'instruction, and it receives a score of 5 out of '
'5. '

In [11]:
# create a dataframe
df = pd.DataFrame({
  "description": descriptions,
  "code": code_samples,
  "reasoning_scores": reasoining_scores
})
# to excel
# add /n every 80 characters to each description
df["description"] = df["description"].apply(lambda x: "\n".join([x[i:i+80] for i in range(0, len(x), 80)]))
# same to the reasoning scores
df["reasoning_scores"] = df["reasoning_scores"].apply(lambda x: "\n".join([x[i:i+80] for i in range(0, len(x), 80)]))

df.to_excel("reasoning_scores.xlsx", index=False)